# Task 4 — Part 2: Decision Tree & Random Forest Regression
**Датасет:** `housing.csv` (Калифорнийская жилая недвижимость, 20 640 домов)  
**Цель:** предсказать `median_house_value` (стоимость жилья)

## 1. Импорты

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Загрузка данных и EDA

In [ ]:
df = pd.read_csv('housing.csv')
df.head(5)

In [ ]:
df.info()

In [ ]:
print("Пропущенные значения:")
print(df.isnull().sum())

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['median_house_value'], bins=50, kde=True, color='steelblue')
plt.title('Распределение стоимости жилья (до очистки)')
plt.xlabel('median_house_value')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes('number').corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Матрица корреляций (числовые признаки)')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='median_income', y='median_house_value', alpha=0.3)
plt.title('Зависимость стоимости от медианного дохода')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='ocean_proximity', y='median_house_value', palette='Set2')
plt.title('Стоимость жилья по близости к океану')
plt.xticks(rotation=15)
plt.show()

## 3. Предобработка данных

In [ ]:
# Заполняем 207 пропусков в total_bedrooms медианой
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

# One-hot encoding категориального признака ocean_proximity
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)
df = df.astype(float)

print("После кодирования — новые колонки:", list(df.columns))

In [ ]:
# Feature engineering: создаём более информативные признаки
df['rooms_per_household']      = df['total_rooms']    / df['households']
df['bedrooms_per_room']        = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population']     / df['households']

# Убираем выбросы: дома с ценой на потолке (500 000 = capped значение в данных)
df = df[df['median_house_value'] < 500_000]

# Исходные «сырые» счётчики теперь заменены производными — удаляем
df = df.drop(columns=['total_rooms', 'total_bedrooms', 'population', 'households'])

print(f"Размер после очистки: {df.shape}")
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Матрица корреляций после Feature Engineering')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['median_house_value'], bins=50, kde=True, color='coral')
plt.title('Распределение стоимости жилья (после очистки)')
plt.show()

## 4. Разделение данных

> **Почему нет StandardScaler?**  
> Деревья решений и случайный лес делают разбиения по порогу (`median_income > 3.5`), а не по расстоянию.  
> Масштаб признаков на них не влияет — скейлинг не нужен.

In [ ]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 5. Decision Tree Regressor — подбор параметров (GridSearchCV)

In [ ]:
param_grid_dt = {
    'max_depth':        [3, 5, 7, 10, 15, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'max_features':     [None, 'sqrt', 'log2']
}

grid_dt = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid_dt,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
grid_dt.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_dt.best_params_}")
print(f"RMSE (CV): {np.sqrt(-grid_dt.best_score_):.2f}")

In [ ]:
# График RMSE по глубине дерева (берём лучшее по остальным параметрам)
results_df = pd.DataFrame(grid_dt.cv_results_)
# Заменяем None глубины на 20 для корректной отрисовки
results_df['param_max_depth'] = results_df['param_max_depth'].fillna(20).astype(int)
depth_rmse = results_df.groupby('param_max_depth')['mean_test_score'].max().apply(lambda s: np.sqrt(-s))

plt.figure(figsize=(8, 4))
plt.plot(depth_rmse.index, depth_rmse.values, marker='o', color='steelblue')
plt.xlabel('max_depth (20 = None)')
plt.ylabel('RMSE (CV)')
plt.title('Подбор глубины дерева (GridSearchCV)')
plt.grid()
plt.show()

## 6. Decision Tree — оценка модели

In [ ]:
dt = grid_dt.best_estimator_
dt_pred = dt.predict(X_test)

dt_mae  = mean_absolute_error(y_test, dt_pred)
dt_mse  = mean_squared_error(y_test, dt_pred)
dt_rmse = np.sqrt(dt_mse)
dt_r2   = r2_score(y_test, dt_pred)

print("=== Decision Tree Regressor ===")
print(f"MAE  (Средняя абсолютная ошибка): {dt_mae:>12.2f}")
print(f"MSE  (Среднеквадратичная ошибка): {dt_mse:>12.2f}")
print(f"RMSE (Корень из MSE):             {dt_rmse:>12.2f}")
print(f"R²   (Коэф. детерминации):        {dt_r2:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) Actual vs Predicted
ax = axes[0]
ax.scatter(y_test, dt_pred, alpha=0.3, s=10)
lim = [y_test.min(), y_test.max()]
ax.plot(lim, lim, 'r--')
ax.set_xlabel('Реальные значения')
ax.set_ylabel('Предсказанные значения')
ax.set_title('Decision Tree: реальные vs предсказанные')

# 2) Residuals
residuals_dt = y_test - dt_pred
ax2 = axes[1]
ax2.scatter(dt_pred, residuals_dt, alpha=0.3, s=10)
ax2.axhline(0, color='red', linestyle='--')
ax2.set_xlabel('Предсказанные значения')
ax2.set_ylabel('Остатки (y - ŷ)')
ax2.set_title('Decision Tree: график остатков')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(residuals_dt, bins=50, kde=True, color='steelblue')
plt.axvline(0, color='red', linestyle='--')
plt.title('Распределение остатков — Decision Tree')
plt.xlabel('Остаток')
plt.show()

In [ ]:
# Визуализация структуры дерева (глубина 3 для читаемости)
dt_vis = DecisionTreeRegressor(max_depth=3, random_state=42)
dt_vis.fit(X_train, y_train)

plt.figure(figsize=(22, 8))
plot_tree(
    dt_vis,
    feature_names=list(X.columns),
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Дерево решений (max_depth=3, для наглядности)')
plt.show()

## 7. Random Forest Regressor — подбор числа деревьев (GridSearchCV)

In [ ]:
param_grid_rf = {'n_estimators': [10, 50, 100, 200, 300]}

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)

best_n = grid_rf.best_params_['n_estimators']
print(f"Лучшее количество деревьев: {best_n}")
print(f"RMSE (CV): {np.sqrt(-grid_rf.best_score_):.2f}")

# График RMSE по числу деревьев
n_estimators_vals = param_grid_rf['n_estimators']
cv_rmse = [np.sqrt(-s) for s in grid_rf.cv_results_['mean_test_score']]

plt.figure(figsize=(8, 4))
plt.plot(n_estimators_vals, cv_rmse, marker='o', color='green')
plt.xlabel('n_estimators')
plt.ylabel('RMSE (CV)')
plt.title('Подбор числа деревьев в Random Forest (GridSearchCV)')
plt.grid()
plt.show()

## 8. Random Forest — оценка модели

In [ ]:
rf = grid_rf.best_estimator_
rf_pred = rf.predict(X_test)

rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_mse  = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2   = r2_score(y_test, rf_pred)

print("=== Random Forest Regressor ===")
print(f"MAE  (Средняя абсолютная ошибка): {rf_mae:>12.2f}")
print(f"MSE  (Среднеквадратичная ошибка): {rf_mse:>12.2f}")
print(f"RMSE (Корень из MSE):             {rf_rmse:>12.2f}")
print(f"R²   (Коэф. детерминации):        {rf_r2:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y_test, rf_pred, alpha=0.3, s=10, color='green')
lim = [y_test.min(), y_test.max()]
ax.plot(lim, lim, 'r--')
ax.set_xlabel('Реальные значения')
ax.set_ylabel('Предсказанные значения')
ax.set_title('Random Forest: реальные vs предсказанные')

residuals_rf = y_test - rf_pred
ax2 = axes[1]
ax2.scatter(rf_pred, residuals_rf, alpha=0.3, s=10, color='green')
ax2.axhline(0, color='red', linestyle='--')
ax2.set_xlabel('Предсказанные значения')
ax2.set_ylabel('Остатки (y - ŷ)')
ax2.set_title('Random Forest: график остатков')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(residuals_rf, bins=50, kde=True, color='green')
plt.axvline(0, color='red', linestyle='--')
plt.title('Распределение остатков — Random Forest')
plt.xlabel('Остаток')
plt.show()

## 9. Важность признаков (Feature Importance)

In [ ]:
importances = rf.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 5))
plt.bar(range(len(importances)), importances[indices], color='steelblue')
plt.xticks(range(len(importances)), feature_names[indices], rotation=45, ha='right')
plt.title('Важность признаков (Random Forest Regressor)')
plt.tight_layout()
plt.show()

print(f"\nСамый важный признак: '{feature_names[indices[0]]}'  (важность = {importances[indices[0]]:.4f})")
print("\nРейтинг всех признаков:")
for rank, idx in enumerate(indices, 1):
    print(f"  {rank:2d}. {feature_names[idx]:<35} {importances[idx]:.4f}")

## 10. Сравнение моделей

In [ ]:
comparison = pd.DataFrame({
    'Модель': ['Decision Tree', 'Random Forest'],
    'MAE':    [dt_mae,  rf_mae],
    'MSE':    [dt_mse,  rf_mse],
    'RMSE':   [dt_rmse, rf_rmse],
    'R²':     [dt_r2,   rf_r2]
})
comparison = comparison.set_index('Модель')
comparison.style.format('{:.2f}').highlight_min(axis=0, color='lightgreen')

In [ ]:
metrics = ['MAE', 'RMSE']
x = np.arange(len(metrics))
width = 0.3

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, [dt_mae, dt_rmse], width, label='Decision Tree', color='steelblue')
ax.bar(x + width/2, [rf_mae, rf_rmse], width, label='Random Forest', color='green')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Ошибка ($)')
ax.set_title('Сравнение MAE и RMSE: Decision Tree vs Random Forest')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Предсказание стоимости нового дома

In [ ]:
# Новый дом: longitude, latitude, housing_median_age, median_income,
# ocean_proximity_INLAND, ocean_proximity_ISLAND, ocean_proximity_NEAR BAY,
# ocean_proximity_NEAR OCEAN, rooms_per_household, bedrooms_per_room, population_per_household

new_house = {
    'longitude':                   -118.5,
    'latitude':                      34.0,
    'housing_median_age':            20.0,
    'median_income':                  5.0,
    'ocean_proximity_INLAND':         0.0,
    'ocean_proximity_ISLAND':         0.0,
    'ocean_proximity_NEAR BAY':       0.0,
    'ocean_proximity_NEAR OCEAN':     1.0,
    'rooms_per_household':            5.5,
    'bedrooms_per_room':              0.18,
    'population_per_household':       2.8
}

new_house_df = pd.DataFrame([new_house], columns=X.columns)

dt_new  = dt.predict(new_house_df)[0]
rf_new  = rf.predict(new_house_df)[0]

print(f"Decision Tree  — прогноз стоимости: ${dt_new:,.0f}")
print(f"Random Forest  — прогноз стоимости: ${rf_new:,.0f}")